<a href="https://colab.research.google.com/github/SujalGodse/Advance-Analytics-using-Statistics/blob/main/Advance_Analytics_using_Statistics_Day19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
  import statsmodels.api as sm
from statsmodels.formula.api import ols
import statsmodels.stats.multicomp
import sklearn
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression


import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
from numpy import random
from collections import Counter
from numpy.linalg import inv
from numpy.linalg import eig
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import pylab
from pylab import legend
from pylab import plot, show, title, xlabel, ylabel
import scipy
from scipy import stats
from scipy.stats import binom,poisson,norm,t,expon,f
from sklearn.model_selection import train_test_split

import statsmodels
from statsmodels import stats
from statsmodels.stats import weightstats as ssw

import statsmodels.api as sm
from statsmodels.formula.api import ols
import statsmodels.stats.multicomp

from statsmodels.discrete.discrete_model import MNLogit

from sklearn.tree import DecisionTreeClassifier

import statsmodels.stats
from statsmodels.stats.proportion import proportions_ztest

from scipy.stats import chisquare

from scipy.stats import chi2_contingency

from statsmodels.stats import rates
from statsmodels.stats.rates import test_poisson

from statsmodels.stats.rates import test_poisson_2indep

from scipy.stats import chi2

# **Dicision Tree**

- it can be used for both regression and classification.

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/CDAC/Titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### **DF Cleaning**

In [ ]:
df = df.drop(['PassengerId','Name','Ticket','Cabin'], axis=1)
print(df.info())
df.Age = df.Age.fillna(df.Age.median())
print(df.Embarked.mode())
df.Embarked = df.Embarked.fillna('S')

df['NumPass'] = df.SibSp + df.Parch
df = df.drop(['SibSp','Parch'], axis=1)

print(df)

dum_Pclass = pd.get_dummies(df.Pclass,drop_first=True,prefix='Pclass').astype(int)
dum_sex = pd.get_dummies(df.Sex,drop_first=True,prefix='Gender').astype(int)
dum_embarked = pd.get_dummies(df.Embarked,drop_first=True,prefix='Embark').astype(int)

df = df.drop(['Pclass','Sex','Embarked'],axis=1)
df = pd.concat([df,dum_Pclass,dum_sex,dum_embarked], axis=1)
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB
None
0    S
Name: Embarked, dtype: object
     Survived  Pclass     Sex   Age     Fare Embarked  NumPass
0           0       3    male  22.0   7.2500        S        1
1           1       1  female  38.0  71.2833        C        1
2           1       3  female  26.0   7.9250        S        0
3           1       1  female  35.0  53.1000        S        1
4           0       3    male  35.0   8.0500        S        0
..        ...     ...  

,Survived,Age,Fare,NumPass,Pclass_2,Pclass_3,Gender_male,Embark_Q,Embark_S
0,0,22.0,7.2500,1,0,1,1,0,1
1,1,38.0,71.2833,1,0,0,0,0,0
2,1,26.0,7.9250,0,0,1,0,0,1
3,1,35.0,53.1000,1,0,0,0,0,1
4,0,35.0,8.0500,0,0,1,1,0,1
...,...,...,...,...,...,...,...,...,...
886,0,27.0,13.0000,0,1,0,1,0,1
887,1,19.0,30.0000,0,0,0,0,0,1
888,0,28.0,23.4500,3,0,1,0,0,1
889,1,26.0,30.0000,0,0,0,1,0,0


In [ ]:
x = df.drop('Survived', axis=1)
y = df.Survived
print(x)
print(y)

      Age     Fare  NumPass  Pclass_2  Pclass_3  Gender_male  Embark_Q  \
0    22.0   7.2500        1         0         1            1         0   
1    38.0  71.2833        1         0         0            0         0   
2    26.0   7.9250        0         0         1            0         0   
3    35.0  53.1000        1         0         0            0         0   
4    35.0   8.0500        0         0         1            1         0   
..    ...      ...      ...       ...       ...          ...       ...   
886  27.0  13.0000        0         1         0            1         0   
887  19.0  30.0000        0         0         0            0         0   
888  28.0  23.4500        3         0         1            0         0   
889  26.0  30.0000        0         0         0            1         0   
890  32.0   7.7500        0         0         1            1         1   

     Embark_S  
0           1  
1           0  
2           1  
3           1  
4           1  
..        ...  

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=20)

from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()

dtc.fit(x_train,y_train)

DecisionTreeClassifier()

In [ ]:
dtc.score(x_train,y_train)  # Percentage Accuracy on training set

0.9845505617977528

In [ ]:
dtc.score(x_test,y_test)  # Percentage Accuracy on testing set

0.7932960893854749

from above we can see there is overfitting. the model has learned noise factors. the bias has been reduced and variance is increased

In [ ]:
dtc = DecisionTreeClassifier(max_depth=3)   # hyperparameter is applied

dtc.fit(x_train,y_train)

print(dtc.score(x_train,y_train))

print(dtc.score(x_test,y_test))

0.8258426966292135
0.8156424581005587


hyperparameter : https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

In [ ]:
dtc = DecisionTreeClassifier()

param = {'max_depth':[2,3,4,5],
         'splitter':['best','random'],
         'criterion':['gini','entropy'],
         'min_samples_leaf':[2,3,4,5,6]}

from sklearn.model_selection import GridSearchCV

mod1 = GridSearchCV(dtc,param_grid=param,cv=5,scoring='accuracy')
mod1.fit(x_train,y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 3, 4, 5],
                         'min_samples_leaf': [2, 3, 4, 5, 6],
                         'splitter': ['best', 'random']},
             scoring='accuracy')

In [ ]:
mod1.best_params_

{'criterion': 'gini',
 'max_depth': 5,
 'min_samples_leaf': 5,
 'splitter': 'best'}

In [ ]:
param = {'max_depth':[2,3,4,5,6,7,8],
         'splitter':['best','random'],
         'criterion':['gini','entropy'],
         'min_samples_leaf':[2,3,4,5,6]}

from sklearn.model_selection import GridSearchCV

mod1 = GridSearchCV(dtc,param_grid=param,cv=5,scoring='accuracy')
mod1.fit(x_train,y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 3, 4, 5, 6, 7, 8],
                         'min_samples_leaf': [2, 3, 4, 5, 6],
                         'splitter': ['best', 'random']},
             scoring='accuracy')

In [ ]:
mod1.best_params_

{'criterion': 'gini',
 'max_depth': 7,
 'min_samples_leaf': 6,
 'splitter': 'best'}

In [ ]:
dtc = DecisionTreeClassifier(criterion='gini',max_depth=5,min_samples_leaf=2,splitter='best')

dtc.fit(x_train,y_train)

print(dtc.score(x_train,y_train))

print(dtc.score(x_test,y_test))

0.8469101123595506
0.8156424581005587


In [ ]:
# df = pd.read_csv('titanic.csv')

# df = df.drop(['PassengerId','Name','Ticket','Cabin'], axis=1)
# df.info()
# df.Age = df.Age.fillna(df.Age.median())
# df.Embarked.mode()
# df.Embarked = df.Embarked.fillna('S')

# df['NumPass'] = df.SibSp + df.Parch
# df = df.drop(['SibSp','Parch'], axis=1)

# dum_Pclass = pd.get_dummies(df.Pclass,drop_first=True,prefix='Pclass').astype(int)
# dum_sex = pd.get_dummies(df.Sex,drop_first=True,prefix='Gender').astype(int)
# dum_embarked = pd.get_dummies(df.Embarked,drop_first=True,prefix='Embark').astype(int)

# df = df.drop(['Pclass','Sex','Embarked'],axis=1)
# df = pd.concat([df,dum_Pclass,dum_sex,dum_embarked], axis=1)

# x=df.drop('Survived',axis=1)
# y=df.Survived

# x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=20)

# from sklearn.tree import DecisionTreeClassifier

# # define instance for DecisionTreeClassifier WITHOUT hyperparameters
# dtc = DecisionTreeClassifier()
# dtc.fit(x_train,y_train)
# dtc.score(x_train,y_train)   # %age accuracy on training set
# dtc.score(x_test,y_test)
# # the big gap between training and test accuracy suggests overfitting

# # to remove overfitting, hyper parameter values need to be set
# # to get the best values for the the hyperparameters we do the Grid Search CV

# param = {'max_depth':[2,3,4,5,6,7,8],
#          'splitter':['best','random'],
#          'criterion':['gini','entropy'],
#          'min_samples_leaf':[2,3,4,5,6]}

# from sklearn.model_selection import GridSearchCV

# mod1 = GridSearchCV(dtc,param_grid=param,cv=5,scoring='accuracy')
# mod1.fit(x_train,y_train)

# mod1.best_params_

# # the above command gives the most appropriate values for the different hyperparameters

# # define the instance dtc WITH hyperparameters
# dtc = DecisionTreeClassifier(criterion='gini',
#  max_depth=5,
#  min_samples_leaf=2,
#  splitter='best')

# dtc.fit(x_train,y_train)
# dtc.score(x_train,y_train)   # %age accuracy on training set
# dtc.score(x_test,y_test)

# y_pred = dtc.predict(x_test)

# print(confusion_matrix(y_test,y_pred))

---

# **Decision Tree using Regression**

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/CDAC/SalesData.xlsx')
df.head()

,Sales,Ad_Budget,Store_Size,Region
0,268.093732,44,2836,East
1,441.660841,96,2656,North
2,347.500444,76,615,North
3,297.618111,64,2036,West
4,203.849511,24,2211,East


In [ ]:
dum_Region = pd.get_dummies(df.Region,drop_first=True,prefix='Reg').astype(int)
df = df.drop('Region', axis=1)
df = pd.concat([df,dum_Region], axis=1)

x = df.drop('Sales', axis=1)
y = df.Sales

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=20)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# define instance for DecisionTreeClassifier WITHOUT hyperparameters
dtr = DecisionTreeRegressor()
dtr.fit(x_train,y_train)

DecisionTreeRegressor()

In [ ]:
dtr.score(x_train,y_train)  # gives the R2 value for the training set

1.0

In [ ]:
dtr.score(x_test,y_test)  # gives the R2 value for the testing set

0.8514599058534646

In [ ]:
from sklearn.metrics import r2_score

y_pred = dtr.predict(x_test)
r2_score(y_test,y_pred)

0.8514599058534646

In [ ]:
# hyperparameter tuning

params = {'max_depth':[3,4,5,6,7],
          'min_samples_split':[2,3,4,5],
          'min_samples_leaf':[2,3,4,5]}

mod1 = GridSearchCV(dtr,param_grid=params,cv=5,scoring='accuracy')
mod1.fit(x_train,y_train)

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(),
             param_grid={'max_depth': [3, 4, 5, 6, 7],
                         'min_samples_leaf': [2, 3, 4, 5],
                         'min_samples_split': [2, 3, 4, 5]},
             scoring='accuracy')

In [ ]:
mod1.best_params_

{'max_depth': 3, 'min_samples_leaf': 2, 'min_samples_split': 2}

In [ ]:
dtr = DecisionTreeRegressor(max_depth=3,min_samples_split=2,min_samples_leaf=2)
dtr.fit(x_train,y_train)
print(dtr.score(x_train,y_train))
print(dtr.score(x_test,y_test))

0.8368972794176226
0.8058459798835955


In [ ]:
1-binom.cdf(9,15,0.7)

np.float64(0.7216214402043639)